# Medical Terminology Matching Using Transformer Models
## Research Experiments, Embedding Extraction, and Evaluation

**Author:** Senior AI/ML Engineer & NLP Researcher  
**Project:** Semantic Similarity-Based Medical Terminology Matching  

### Abstract
This research notebook evaluates semantic similarity between medical terminology pairs using:
1. **Sentence-BERT (`all-MiniLM-L6-v2`)**
2. **BioBERT (PubMed Contextual Model with Attention-Mask-Aware Mean Pooling)**
3. **Traditional Baseline Methods (TF-IDF + Cosine, Jaccard, Levenshtein Distance)**
4. **Dense Vector Search with FAISS (Inner Product Cosine Similarity)**

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

from models.sentence_bert import SentenceBERTModel
from models.biobert import BioBERTModel
from models.traditional_similarity import TraditionalSimilarityService
from services.faiss_service import FAISSService
from services.evaluation_service import EvaluationService

print("Environment initialized successfully!")
print("PyTorch version:", torch.__version__)

## 1. Dataset Loading and Inspection
Load the curated medical terms dataset (130 terms across 14 specialties) and labeled evaluation pairs.

In [ ]:
terms_df = pd.read_csv('../data/medical_terms.csv')
eval_df = pd.read_csv('../data/evaluation_pairs.csv')

print(f"Total indexed terminology records: {len(terms_df)}")
print(f"Total evaluation pairs: {len(eval_df)}")
terms_df.head(5)

## 2. Model Inference & Semantic Proximity Comparison
Compare canonical clinical synonyms (e.g. Heart Attack vs Myocardial Infarction).

In [ ]:
sbert = SentenceBERTModel.get_instance()
trad = TraditionalSimilarityService()

term_a = "Heart Attack"
term_b = "Myocardial Infarction"

sbert_sim = sbert.compute_similarity(term_a, term_b)
trad_metrics = trad.compare_terms(term_a, term_b)

print(f"Pair: '{term_a}' ↔ '{term_b}'")
print(f"Sentence-BERT Cosine Similarity: {sbert_sim:.4f}")
print(f"TF-IDF Cosine Similarity:         {trad_metrics['tfidf_cosine']:.4f}")
print(f"Jaccard Token Similarity:         {trad_metrics['jaccard_word']:.4f}")
print(f"Levenshtein Similarity:           {trad_metrics['levenshtein_similarity']:.4f}")

## 3. Empirical Evaluation on Labeled Test Partition
Calculate Accuracy, Precision, Recall, F1-Score, and ROC-AUC on unseen test pairs.

In [ ]:
eval_svc = EvaluationService()
metrics = eval_svc.evaluate_model('Sentence-BERT', split='test', binary_threshold=0.60)

print("=== Empirical Evaluation Results (Test Partition) ===")
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print(f"F1-Score:  {metrics['f1_score']:.4f}")
print(f"ROC-AUC:   {metrics['roc_auc']:.4f}")
print("Confusion Matrix:", metrics['confusion_matrix'])
print("Precision@K:", metrics['precision_at_k'])
print("Recall@K:", metrics['recall_at_k'])
print("Leakage Audit:", metrics['data_leakage_audit']['has_leakage'])

## 4. FAISS Sub-Millisecond Dense Vector Search
Execute nearest-neighbor semantic search over medical terminology index.

In [ ]:
faiss_svc = FAISSService.get_instance()
query = "High Blood Pressure"
results = faiss_svc.search(query, top_k=5)

print(f"Top-5 matches for query: '{query}'")
for r in results:
    print(f"Rank {r['rank']}: {r['medical_term']} ({r['category']}) - Cosine: {r['similarity_score']:.4f} [{r['relationship']}]")